In [47]:
%pip install --upgrade timm torch torchvision
%pip install timm==0.9.12

import torch
import torchvision
import torchvision.transforms as transforms
from torchvision import models
import timm
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 2.1 MB/s  0:00:01 eta 0:00:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.24
    Uninstalling timm-1.0.24:
      Successfully uninstalled timm-1.0.24
Note: you may need to restart the kernel to use updated packages.


In [48]:
import torch 
import torch as nn 
import torch.optim as optim

import time 
import math 
import random
import numpy as np

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [49]:
device="cuda" if torch.cuda.is_available() else "cpu"

In [54]:
cvt_models=timm.list_models("cvt*")
print(cvt_models)


[]


In [55]:
cvt_models[:20]

[]

In [56]:
len(cvt_models)

0

In [57]:
def pick_model_name(names):
    prefs=['cvt_13','cvt-13','cvt13','cvt_21','cvt-21','cvt21']
    for p in prefs:
        for n in names:
            if n.startswith(p):
                return n
    return names[0] if names else None

model_name = pick_model_name(cvt_models)

if model_name is None:
    print('Warning: No CvT model found. Using a default model.')
    model_name = 'cvt_13'

print(f'Selected model: {model_name}')

Selected model: cvt_13


In [59]:
IMG_SIZE=224
BATCH_SIZE=64 
NUM_WORKERS=2
EPOCHS=5

train_tfms=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [60]:
test_tfms=transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914,0.4812,0.4465),std=(0.2470,0.2435,0.2616))
])

In [61]:
data_root="./data"
train_ds=datasets.CIFAR10(root=data_root,train=True,download=True,transform=train_tfms)
test_ds=datasets.CIFAR10(root=data_root,train=True,download=True,transform=test_tfms)

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True)
len(train_ds)
len(test_ds)

Files already downloaded and verified
Files already downloaded and verified


50000

In [62]:
len(test_ds)

50000

In [63]:
# Use a standard Vision Transformer model since CvT models are not available
model_name = 'vit_tiny_patch16_224'

model = timm.create_model(model_name, pretrained=True, num_classes=10)
model = model.to(device)
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    y = model(x)
    print(f'Model output shape: {y.shape}')

Model output shape: torch.Size([2, 10])


In [64]:
y.shape

torch.Size([2, 10])

In [65]:
def accracy_score(logits,targets):
    preds=logits.argmax(dim=1)
    return (preds==targets).float().mean().item()
def train_one_epoch(model,loader,oprtimizer,criterion,epoch=0):
    model.train()
    running_loss=0.0
    running_acc=0.0
    for step,(image,labels) in enumerate(loader):
        images,labels=images.to(device,no_blocking=True),labels.to(device,no_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits=model(images)
        loss=criterion(logits,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        running_acc  += accuracy_top1(logits.detach(),labels)
    return running_loss/len(loader),running_acc/len(loader)